In [ ]:
import pandas as pd
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments




In [ ]:
#load the 1,383 filtered synthetic pairs form the generation notebook

stage1 = pd.read_csv("../data/stage1_train.csv")

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
#BART-base: pretrained encoder-decoder, ~140M parameters.
#I choose it as it is a standard seq2seq model for summarization/simplification that fits my 6 GB GPU limitation

#checking the length of all the pairs
input_lengths = []
for text in stage1["clause_text"]:
    tokens = tokenizer(text)["input_ids"]
    input_lengths.append(len(tokens))


target_lengths = []
for text in stage1["rewrite"]:
    tokens = tokenizer(text)["input_ids"]
    target_lengths.append(len(tokens))


stage1["input_tokens"] = input_lengths
stage1["target_tokens"] = target_lengths


print("len input")
print(stage1["input_tokens"].describe())
print()
print("len target")
print(stage1["target_tokens"].describe())
print()



In [ ]:
#checking how many pairs exceed the 512 input and 256 target tokens
too_long_inputs = 0
for length in input_lengths:
    if length > 512:
        too_long_inputs = too_long_inputs + 1

too_long_targets = 0
for length in target_lengths:
    if length > 256:
        too_long_targets = too_long_targets + 1

print(too_long_inputs, "more than 512 ")
print(too_long_targets, "more than 256")

In [ ]:
stage1 = stage1[(stage1["input_tokens"] <= 512) & (stage1["target_tokens"] <= 256)]
stage1.to_csv("../datav2/stage1_train_final.csv", index=False)
# removing the pairs that were too long (the number was small enough to justify this)

In [ ]:
#some markers where not cleaned before so I cleaned them here
set_temp = pd.read_csv("../data/cuad_train.csv")
cuad_test = pd.read_csv("../data/cuad_test.csv")

print(set_temp["clause_text"].str.contains("* * *", regex=False).sum(), "[* * *]")
print(cuad_test["clause_text"].str.contains("* * *", regex=False).sum(), "[* * *]")
print(set_temp["clause_text"].str.contains("XXXX", regex=False).sum(), "XXXX")
print(cuad_test["clause_text"].str.contains("XXXX", regex=False).sum(), "XXXX")

In [ ]:
contracts = stage1["contract"].drop_duplicates()
val_contracts = contracts.sample(frac=0.1, random_state=1)

is_val = stage1["contract"].isin(val_contracts)
stage1_val = stage1[is_val].copy()
stage1_train = stage1[~is_val].copy()


print("train:", len(stage1_train))
print("val: ", len(stage1_val))
#10 % of contracts become validation data

In [ ]:
def temp(text):
    if "* * *" in text:
        return True
    if "XXXX" in text:
        return True
    return False

bad_train = stage1_train["clause_text"].apply(temp) | stage1_train["rewrite"].apply(temp)
bad_val = stage1_val["clause_text"].apply(temp) | stage1_val["rewrite"].apply(temp)

print(bad_train.sum(), "remocved from train")
print(bad_val.sum(), "removed from val")

stage1_train = stage1_train[~bad_train]
stage1_val = stage1_val[~bad_val]

#for reproducibility: I removed the 13 marker pairs after the original runs -> the training is unaffected but the validation data might slightly differ


In [ ]:

stage1_train.to_csv("../datav2/stage1_train_split.csv", index=False)
stage1_val.to_csv("../datav2/stage1_val_split.csv", index=False)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(stage1_train[["clause_text", "rewrite"]])
val_ds = Dataset.from_pandas(stage1_val[["clause_text", "rewrite"]])


def preprocess(example):
    model_inputs = tokenizer(example["clause_text"])
    labels = tokenizer(text_target=example["rewrite"])
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
train_tok = train_ds.map(preprocess, remove_columns=["clause_text", "rewrite", "__index_level_0__"])
val_tok = val_ds.map(preprocess, remove_columns=["clause_text", "rewrite","__index_level_0__"])
print(train_tok)
print(val_tok)

#DONT RUN!!

from transformers import AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer

#stage 1
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(output_dir="../models/stage1",  #as a checkpoint after each epoch
                        num_train_epochs=4,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",      # val loss atfer each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,    # keep the model with the least loss in val
    fp16=True,
    report_to="none")

trainer = Seq2SeqTrainer(model=model,
                         args=args,
                         train_dataset=train_tok,
                         eval_dataset=val_tok,
                         data_collator=data_collator)

trainer.train()

trainer.save_model("../models/stage1_best")
tokenizer.save_pretrained("../models/stage1_best")


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_path = "../models/stage1_best"
test_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
test_tokenizer = AutoTokenizer.from_pretrained(model_path)

#testing the model
for clause in stage1_val["clause_text"].head(10):
    inputs = test_tokenizer(clause, return_tensors="pt", max_length=512, truncation=True)
    output = test_model.generate(**inputs, max_length=256, num_beams=4)
    rewrite = test_tokenizer.decode(output[0], skip_special_tokens=True)

    print("ORIGINAL:", clause)
    print("MODEL:   ", rewrite)
    print()

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

#stage 2 tokanization
tos_train = pd.read_csv("../data/tos_train.csv")
tokenizer = AutoTokenizer.from_pretrained("../models/stage1_best")

input_lengths = []
for text in tos_train["original_text"]:
    tokens = tokenizer(text)["input_ids"]
    input_lengths.append(len(tokens))

target_lengths = []
for text in tos_train["reference_summary"]:
    tokens = tokenizer(text)["input_ids"]
    target_lengths.append(len(tokens))



tos_train["input_tokens"] = input_lengths
tos_train["target_tokens"] = target_lengths


print(tos_train["input_tokens"].describe())

print(tos_train["target_tokens"].describe())



In [ ]:

too_long_inputs = 0
for length in input_lengths:
    if length > 512:
        too_long_inputs = too_long_inputs + 1

too_long_targets = 0
for length in target_lengths:
    if length > 256:
        too_long_targets = too_long_targets + 1

print(too_long_inputs, "more than 512 ")
print(too_long_targets, "more than 256")

In [ ]:

#cutting off the inputs that are too long (same as above)
stage2 = tos_train[(tos_train["input_tokens"] <= 512) ]

tos_val = stage2.sample(frac=0.1, random_state=1)
tos_train_final = stage2.drop(tos_val.index)

print("train", len(tos_train_final))
print("val ", len(tos_val))

tos_train_final.to_csv("../datav2/stage2_train_split.csv", index=False)
tos_val.to_csv("../datav2/stage2_val_split.csv", index=False)

In [ ]:
from datasets import Dataset

train_ds2 = Dataset.from_pandas(tos_train_final[["original_text", "reference_summary"]])
val_ds2 = Dataset.from_pandas(tos_val[["original_text", "reference_summary"]])

def preprocess2(example):
    model_inputs = tokenizer(example["original_text"])
    labels = tokenizer(text_target=example["reference_summary"])
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_tok2 = train_ds2.map(preprocess2, remove_columns=["original_text", "reference_summary", "__index_level_0__"])
val_tok2 = val_ds2.map(preprocess2, remove_columns=["original_text", "reference_summary", "__index_level_0__"])
print(train_tok2)
print(val_tok2)

#phase 2

model2 = AutoModelForSeq2SeqLM.from_pretrained("../models/stage1_best")  #taking the best model from phase 1 and building uppon it
data_collator2 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model2)

args2 = Seq2SeqTrainingArguments(
    output_dir="../models/stage2",
    num_train_epochs=8,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    fp16=True,
    report_to="none")

trainer2 = Seq2SeqTrainer(
    model=model2,
    args=args2,
    train_dataset=train_tok2,
    eval_dataset=val_tok2,
    data_collator=data_collator2)

trainer2.train()

trainer2.save_model("../models/stage2_best")
tokenizer.save_pretrained("../models/stage2_best")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_path = "../models/stage2_best"
test_model2 = AutoModelForSeq2SeqLM.from_pretrained(model_path)
test_tokenizer2 = AutoTokenizer.from_pretrained(model_path)

#testing the model
for i, row in tos_val.head(5).iterrows():
    inputs = test_tokenizer2(row["original_text"], return_tensors="pt", max_length=512, truncation=True)
    output = test_model2.generate(**inputs, max_length=256, num_beams=4)
    summary = test_tokenizer2.decode(output[0], skip_special_tokens=True)

    print("ORIGINAL: ", row["original_text"])
    print("REFERENCE:", row["reference_summary"])
    print("MODEL:    ", summary)
    print()

#base model trained on only the ToS;DR data
model_only2 = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
data_collator_only2 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_only2)

args_only2 = Seq2SeqTrainingArguments(
    output_dir="../models/stage2_only",
    num_train_epochs=8,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    fp16=True,
    report_to="none")

trainer_only2 = Seq2SeqTrainer(
    model=model_only2,
    args=args_only2,
    train_dataset=train_tok2,
    eval_dataset=val_tok2,
    data_collator=data_collator_only2)

trainer_only2.train()

trainer_only2.save_model("../models/stage2_only_best")
tokenizer.save_pretrained("../models/stage2_only_best")

In [ ]:

import torch
#test sets
tos_test = pd.read_csv("../data/tos_test.csv")
cuad_test = pd.read_csv("../data/cuad_test.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"  #if the Gpu is overloaded it should use the cpu


In [ ]:
def load_model(path):
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model = model.to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(path)
    return model, tokenizer


def summarize(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    inputs = inputs.to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, num_beams=4)
    return tokenizer.decode(output[0], skip_special_tokens=True)

models = {
    "two_stage": "../models/stage2_best",
    "stage2_only": "../models/stage2_only_best",
}

for name, path in models.items():
    model, tokenizer = load_model(path)

    tos_outputs = []
    for text in tos_test["original_text"]:
        tos_outputs.append(summarize(model, tokenizer, text))
    tos_test[name] = tos_outputs

    probe_outputs = []
    for text in cuad_test["clause_text"]:
        probe_outputs.append(summarize(model, tokenizer, text))
    cuad_test[name] = probe_outputs

    print(name, "done")

tos_test.to_csv("../results/tos_test_outputs.csv", index=False)
cuad_test.to_csv("../results/probe_outputs.csv", index=False)

In [ ]:
#counting test inuputs longer than 512 tokens, tthey get truncated

count_tos = 0
for text in tos_test["original_text"]:
    if len(tokenizer(text)["input_ids"]) > 512:
        count_tos = count_tos + 1

count_probe = 0
for text in cuad_test["clause_text"]:
    if len(tokenizer(text)["input_ids"]) > 512:
        count_probe = count_probe + 1

print(count_tos, "/", len(tos_test), "tos_test longer -> 512 tokens")
print(count_probe, "/", len(cuad_test), "probe longer -> 512 tokens")

In [ ]:
import pandas as pd

probe_out = pd.read_csv("../results/probe_outputs.csv")

different = probe_out[probe_out["two_stage"] != probe_out["stage2_only"]]
print(len(different), "of", len(probe_out), "clauses get different outputs")


sample = different.sample(10, random_state=1)

for i, row in sample.iterrows():
    print("original:   ", row["clause_text"])
    print("two stages:  ", row["two_stage"])
    print("stage 2 (only):", row["stage2_only"])
    print()

In [ ]:
#scoring
import pandas as pd
from src.faithfulness_filter import model_CZS
from src.readability_filter import readability_drop

tos_out = pd.read_csv("../results/tos_test_outputs.csv")
probe_out = pd.read_csv("../results/probe_outputs.csv")
tos_haiku = pd.read_csv("../results/tos_test_outputs_haiku.csv")
probe_haiku = pd.read_csv("../results/probe_outputs_haiku.csv")

print((tos_out["original_text"] == tos_haiku["original_text"]).all())
print((probe_out["clause_text"] == probe_haiku["clause_text"]).all())

tos_out["haiku"] = tos_haiku["haiku"]
probe_out["haiku"] = probe_haiku["haiku"]

tos_out = tos_out.fillna("")
probe_out = probe_out.fillna("")

In [ ]:
def forward_score(original, output):
    return model_CZS.score([original], [output])["scores"][0]


def length_ratio(original, output):
    return len(output.split()) / len(original.split())


def normalize(text):
    text = text.lower()
    text = text.replace("'", " ")
    text = text.replace("’", " ")
    return text

In [ ]:
tos_systems = ["two_stage", "stage2_only", "reference_summary", "haiku"]

for system in tos_systems:
    grade_drops = []
    forwards = []
    ratios = []

    for i, row in tos_out.iterrows():
        original = row["original_text"]
        output = row[system]
        grade_drops.append(readability_drop(original, output))
        forwards.append(forward_score(original, output))
        ratios.append(length_ratio(original, output))

    tos_out[system + "_grade_drop"] = grade_drops
    tos_out[system + "_forward"] = forwards
    tos_out[system + "_length_ratio"] = ratios
    print(system, "done")

In [ ]:
import evaluate

sari = evaluate.load("sari")

for system in ["two_stage", "stage2_only", "haiku"]:
    scores = []
    for i, row in tos_out.iterrows():
        result = sari.compute(
            sources=[normalize(row["original_text"])],
            predictions=[normalize(row[system])],
            references=[[normalize(row["reference_summary"])]],
        )
        scores.append(result["sari"])
    tos_out[system + "_sari_norm"] = scores
    print(system, "SARI done")

In [ ]:
probe_systems = ["two_stage", "stage2_only", "haiku"]

for system in probe_systems:
    grade_drops = []
    forwards = []
    ratios = []

    for i, row in probe_out.iterrows():
        original = row["clause_text"]
        output = row[system]
        grade_drops.append(readability_drop(original, output))
        forwards.append(forward_score(original, output))
        ratios.append(length_ratio(original, output))

    probe_out[system + "_grade_drop"] = grade_drops
    probe_out[system + "_forward"] = forwards
    probe_out[system + "_length_ratio"] = ratios
    print(system, "done")

In [ ]:
tos_out.to_csv("../results/tos_test_scored_all.csv", index=False)
probe_out.to_csv("../results/probe_scored_all.csv", index=False)
print("saved")

In [ ]:
print("tos test")
for system in tos_systems:
    print(system)
    print("mean grade drop ", round(tos_out[system + "_grade_drop"].mean(), 2))
    print("mean forward:  ", round(tos_out[system + "_forward"].mean(), 3))
    print("share unsupported: ", round((tos_out[system + "_forward"] < 0.16).mean(), 3))
    print("mean length ratio: ", round(tos_out[system + "_length_ratio"].mean(), 2))
print()

print("cuad test")
for system in probe_systems:
    print(system)
    print("mean grade drop:", round(probe_out[system + "_grade_drop"].mean(), 2))
    print("mean forward:", round(probe_out[system + "_forward"].mean(), 3))
    print("share unsupported: ", round((probe_out[system + "_forward"] < 0.16).mean(), 3))
    print("mean length ratio: ", round(probe_out[system + "_length_ratio"].mean(), 2))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tos_out = pd.read_csv("../results/tos_test_scored_all.csv")
probe_out = pd.read_csv("../results/probe_scored_all.csv")

tos_systems = ["two_stage", "stage2_only", "reference_summary", "haiku"]
probe_systems = ["two_stage", "stage2_only", "haiku"]



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tos_out = pd.read_csv("../results/tos_test_scored_all.csv")
probe_out = pd.read_csv("../results/probe_scored_all.csv")

tos_systems = ["two_stage", "stage2_only", "reference_summary", "haiku"]
probe_systems = ["two_stage", "stage2_only", "haiku"]



rows = []
for system in tos_systems:
    rows.append({
        "system": system,
        "grade drop": tos_out[system + "_grade_drop"].mean(),
        "forward": tos_out[system + "_forward"].mean(),
        "unsupported": (tos_out[system + "_forward"] < 0.16).mean(),
        "length ratio": tos_out[system + "_length_ratio"].mean()})
tos_summary = pd.DataFrame(rows).set_index("system")
print(tos_summary)

tos_summary.plot(kind="bar", subplots=True, layout=(1, 4), figsize=(16, 4),
                 legend=False, rot=30, title=list(tos_summary.columns))
plt.tight_layout()
plt.savefig("../results/tos_test_stats.pdf")
plt.show()


In [ ]:
rows = []
for system in probe_systems:
    rows.append({
        "system": system,
        "grade drop": probe_out[system + "_grade_drop"].mean(),
        "forward": probe_out[system + "_forward"].mean(),
        "unsupported": (probe_out[system + "_forward"] < 0.16).mean(),
        "length ratio": probe_out[system + "_length_ratio"].mean()})
probe_summary = pd.DataFrame(rows).set_index("system")
print(probe_summary)

probe_summary.plot(kind="bar", subplots=True, layout=(1, 4), figsize=(16, 4),
                   legend=False, rot=30, title=list(probe_summary.columns))
plt.tight_layout()
plt.savefig("../results/probe_stats.pdf")
plt.show()



In [ ]:
from scipy.stats import shapiro
import matplotlib.pyplot as plt
import pandas as pd

tos = pd.read_csv("../results/tos_test_scored_all.csv")
differences = tos["two_stage_forward"] - tos["stage2_only_forward"]

print("Shapiro-Wilk p:", round(shapiro(differences).pvalue, 4))

plt.hist(differences, bins=30)
plt.title("Differences in forward score (two_stage - stage2_only)")
plt.show()

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon, shapiro

tos = pd.read_csv("../results/tos_test_scored_all.csv")
probe = pd.read_csv("../results/probe_scored_all.csv")


def compare(df, test_set, system_a, system_b, metric):
    a = df[system_a + "_" + metric]
    b = df[system_b + "_" + metric]

    differences = a - b
    normal_p = shapiro(differences).pvalue
    test_p = wilcoxon(a, b).pvalue

    return {
        "test_set": test_set,
        "metric": metric,
        "system_a": system_a,
        "system_b": system_b,
        "mean_a": round(a.mean(), 3),
        "mean_b": round(b.mean(), 3),
        "shapiro_p": round(normal_p, 4),
        "wilcoxon_p": round(test_p, 4),
    }

In [ ]:
pairs = [
    ("two_stage", "stage2_only"),
    ("haiku", "two_stage"),
    ("haiku", "stage2_only"),
]

results = []

for system_a, system_b in pairs:
    results.append(compare(tos, "tos_test", system_a, system_b, "sari_norm"))
    results.append(compare(tos, "tos_test", system_a, system_b,"forward"))
    results.append(compare(tos, "tos_test", system_a, system_b, "grade_drop"))

for system_a, system_b in pairs:
    results.append(compare(probe,"cuad_probe", system_a, system_b,"forward"))
    results.append(compare(probe, "cuad_probe", system_a, system_b, "grade_drop"))

results = pd.DataFrame(results)
results["significant"] = results["wilcoxon_p"]< 0.05/15

results.to_csv("../results/significance_tests.csv", index=False)
results

In [ ]:
import os

files = [
    "../data/tos_train.csv",
    "../data/tos_test.csv",
    "../data/cuad_train.csv",
    "../data/cuad_test.csv",
    "../data/generated_cuad_raw.csv",
    "../data/generated_scored.csv",
    "../data/generated_raw_v2(selfmade_summaries).csv",
    "../data/generated_scored_v2.csv",
    "../data/stage1_train.csv",
    "../datav2/stage1_train_split.csv",
    "../datav2/stage1_val_split.csv",
    "../datav2/stage2_train_split.csv",
    "../datav2/stage2_val_split.csv",
    "../results/tos_test_outputs.csv",
    "../results/probe_outputs.csv",
    "../results/tos_test_outputs_haiku.csv",
    "../results/probe_outputs_haiku.csv",
    "../models/stage1_best",
    "../models/stage2_best",
    "../models/stage2_only_best",
]

for f in files:
    print(os.path.exists(f), f)